# TERA — Language-Table Closed-Loop Task Success Evaluation

Measures **task success rate** (%) for all 6 ablation conditions × 3 seeds  
in the Language-Table simulator.  Runtime: ~25–40 min on T4 GPU.

## Before running
1. **Runtime → Change runtime type → T4 GPU**
2. Mount Drive and upload your checkpoints (see Cell 2 instructions)
3. Run cells top-to-bottom

## Checkpoint folder structure on Drive
```
MyDrive/VERA_LT_Checkpoints/
  lt_full_vera/seed42/best_sft_vera.pt
  lt_full_vera/seed123/best_sft_vera.pt
  lt_full_vera/seed456/best_sft_vera.pt
  lt_bc_baseline/seed42/best_sft_vera.pt    ... etc.
  lt_no_lang/...
  lt_no_act/...
  lt_no_exp/...
  lt_no_hist_tf/...
```
The local 'Test1' folder has `checkpoints N` dirs — reorganise them into the flat structure above before uploading.

In [ ]:
# Cell 1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
# Cell 2 — Clone repo (skip if already on Drive)
import os
REPO_DRIVE = '/content/drive/MyDrive/VLA-Robot-Learning'
if os.path.isdir(REPO_DRIVE):
    print(f'Repo found at {REPO_DRIVE}')
    REPO = REPO_DRIVE
else:
    !git clone https://github.com/YOUR/VLA-Robot-Learning /content/VLA-Robot-Learning
    REPO = '/content/VLA-Robot-Learning'
    print(f'Cloned to {REPO}')

import sys
sys.path.insert(0, REPO)
CONFIG_PATH = f'{REPO}/configs/config.yaml'
print(f'Config: {CONFIG_PATH}')

In [ ]:
# Cell 3 — User Config (edit paths here)
MYDRIVE     = '/content/drive/MyDrive'
CKPT_ROOT   = f'{MYDRIVE}/VERA_LT_Checkpoints'   # ← edit if different
N_EPISODES  = 50       # per condition per seed (50 × 3 seeds = 150 episodes total per condition)
SEEDS       = [42, 123, 456]
MAX_STEPS   = 60
REWARD_MODE = 'block2block'
LT_ACTION_SCALE   = 0.03
SUCCESS_REWARD_THR = 0.15   # fallback if done=True never fires

OUT_JSON = f'{CKPT_ROOT}/lt_task_success_results.json'
OUT_TXT  = f'{CKPT_ROOT}/lt_task_success_results.txt'

# Ablation map: (dir_name, display_name, vera_flag_overrides, suppress_action_tok)
ABLATION_CONDITIONS = [
    ('lt_full_vera',   'Full TERA ★ (all 5 streams)',            {},                                              False),
    ('lt_bc_baseline', 'BC/SFT baseline',                        {'use_lang_feedback': False, 'use_temporal_history': False}, False),
    ('lt_no_lang',     'No lang. feedback (1,2,4)',               {'use_lang_feedback': False},                    False),
    ('lt_no_exp',      'No E_emb — narration only (1,2,3a,4)',    {'use_consequence_token': False},                False),
    ('lt_no_act',      'No E_act — emb. know. only (1,2,3b,4)',   {},                                              True),
    ('lt_no_hist_tf',  'No hist. TF (1,2,3a,3b)',                {'use_temporal_history': False},                 False),
]
print('Config OK')

In [ ]:
# Cell 4 — Install Language-Table + deps
import subprocess, sys
def pip_one(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg], stderr=subprocess.STDOUT)
    print(f'  ✓ {pkg[:60]}')

for pkg in ('pyyaml', 'pillow', 'numpy'):
    pip_one(pkg)
pip_one('gym<=0.23.0')
pip_one('pybullet')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
    'git+https://github.com/google-research/language-table.git'])
print('  ✓ language-table')
try:
    import clip
except ImportError:
    pip_one('git+https://github.com/openai/CLIP.git')

from language_table.environments import language_table as _lt_check
print('\nlanguage_table import OK')

In [ ]:
# Cell 5 — Run evaluation (calls the eval script directly)
# Inject config so the script uses the variables defined above
import importlib.util, types

# Patch globals into the script's namespace
SCRIPT = f'{REPO}/docs/eval_lt_task_success.py'

# Override config variables before running
overrides = dict(
    MYDRIVE=MYDRIVE, CKPT_ROOT=CKPT_ROOT, CONFIG_PATH=CONFIG_PATH,
    N_EPISODES=N_EPISODES, SEEDS=SEEDS, MAX_STEPS=MAX_STEPS,
    REWARD_MODE=REWARD_MODE, LT_ACTION_SCALE=LT_ACTION_SCALE,
    SUCCESS_REWARD_THR=SUCCESS_REWARD_THR,
    ABLATION_CONDITIONS=ABLATION_CONDITIONS,
    OUT_JSON=OUT_JSON, OUT_TXT=OUT_TXT,
)

with open(SCRIPT) as f:
    src = f.read()

# Skip sections 0 and 2 (config/install already done)
# Execute from section 3 onward
exec(compile(src, SCRIPT, 'exec'), {**overrides, '__name__': '__main__'})

In [ ]:
# Cell 6 — View saved results
import json
with open(OUT_JSON) as f:
    res = json.load(f)

print(f'{'Method':<44} {'Success %':>12}  {'±':>3}  {'Reward':>8}')
print('-' * 75)
for k, v in res.items():
    print(f"  {v['display_name']:<42} {v['success_mean']:>6.1f}%  ±{v['success_std']:>4.1f}%  {v['reward_mean']:>6.3f}")